# Data cleaning

Load the latest source exports, preserve untouched raw tables, then validate grain before applying cleaning rules.

In [ ]:
from pathlib import Path
import os

import pandas as pd


# Local data folder. Set this in your shell before launching Jupyter or VS Code.
data_dir = os.environ.get('RETAIL_DATA_DIR')
if not data_dir:
    raise EnvironmentError('Set RETAIL_DATA_DIR to your local Retail Business data folder before running this notebook.')
DATA_DIR = Path(data_dir).expanduser()


def latest_file(folder, suffix):
    """Return the latest ISO-dated export matching the expected suffix."""
    matches = sorted(folder.glob(f"*{suffix}"))
    if not matches:
        raise FileNotFoundError(f"No files matching *{suffix} in {folder}")
    return matches[-1]


folders = {
    "customers": DATA_DIR / "Dim_Customers",
    "orders": DATA_DIR / "Fact_Orders",
    "products": DATA_DIR / "Dim_Products",
    "inventory": DATA_DIR / "Dim_Inventory",
    "variants": DATA_DIR / "Dim_Variants"
}

suffixes = {
    "customers": "_Customers.csv",
    "orders": "_Orders.csv",
    "products": "_Products.csv",
    "inventory": "_Inventory_Items.csv",
    "variants": "_Variants.csv"
}

tables = {}
source_files = {}

for name, folder in folders.items():
    source_file = latest_file(folder, suffixes[name])
    source_files[name] = source_file
    tables[name] = pd.read_csv(source_file)
    print(f"{name}: {source_file.name} — {len(tables[name]):,} raw rows")

In [ ]:
# Preserve untouched source tables. Cleaning will produce separate *_clean tables.
customers_raw = tables["customers"].copy()
orders_raw = tables["orders"].copy()
products_raw = tables["products"].copy()
inventory_raw = tables["inventory"].copy()
variants_raw = tables["variants"].copy()


In [ ]:
# Initial source verification; uniqueness alone does not establish semantic grain.
raw_tables = {
    "customers_raw": (customers_raw, "ID"),
    "orders_raw": (orders_raw, "ID"),
    "products_raw": (products_raw, "ID"),
    "inventory_raw": (inventory_raw, "ID"),
    "variants_raw": (variants_raw, "ID")
}

for name, (df, candidate_key) in raw_tables.items():
    print(
        f"{name}: {len(df):,} rows; "
        f"{df[candidate_key].nunique(dropna=True):,} distinct non-null {candidate_key} values; "
        f"{df[candidate_key].isna().sum():,} nulls"
    )

In [ ]:
# Normalize Shopify GIDs
def extract_shopify_id(series, entity):
    return (
        series.astype("string")
        .str.extract(fr"^gid://shopify/{entity}/(\d+)$", expand=False)
    )

In [ ]:
# Call extract_shopify_id on orders table
orders_work = orders_raw.copy()

orders_work["order_id"] = extract_shopify_id(orders_work["ID"], "Order")

orders_work["customer_id"] = extract_shopify_id(orders_work["Customer: ID"], "Customer")

orders_work["order_line_id"] = extract_shopify_id(orders_work["Line items: ID"], "LineItem")

orders_work["product_id"] = extract_shopify_id(orders_work["Line items / Product: ID"], "Product")

orders_work["variant_id"] = extract_shopify_id(orders_work["Line items / Variant: ID"], "ProductVariant")

orders_work["fulfillment_id"] = extract_shopify_id(orders_work["Fulfillments: ID"], "Fulfillment")


In [ ]:
# Call extract_shopify_id on customers table
customers_work = customers_raw.copy()

customers_work["customer_id"] = extract_shopify_id(customers_work["ID"], "Customer")

In [ ]:
# Call extract_shopify_id on products table
products_work = products_raw.copy()

products_work["product_id"] = extract_shopify_id(products_work["ID"], "Product")

In [ ]:
# Call extract_shopify_id on inventory table
inventory_work = inventory_raw.copy()

inventory_work["inventory_item_id"] = extract_shopify_id(inventory_work["ID"], "InventoryItem")

inventory_work["variant_id"] = extract_shopify_id(inventory_work["Variant: ID"], "ProductVariant")

inventory_work["inventory_level_id"] = inventory_work["Inventory levels: ID"].str.extract(r"gid://shopify/InventoryLevel/(\d+)\?", expand=False,).astype("string")

inventory_work["location_id"] = extract_shopify_id(inventory_work["Inventory levels / Location: ID"], "Location")

# One current inventory level is identified by its inventory item and level component.
inventory_level_pk = ["inventory_item_id", "inventory_level_id"]

In [ ]:
# Call extract_shopify_id on variants table
variants_work = variants_raw.copy()

variants_work["variant_id"] = extract_shopify_id(variants_work["ID"], "ProductVariant")

variants_work["product_id"] = extract_shopify_id(variants_work["Product: ID"], "Product")

variants_work["inventory_item_id"] = extract_shopify_id(variants_work["Inventory item: ID"], "InventoryItem")

In [ ]:
# Validate ordinary Shopify GIDs and display all results before failing.
id_checks = [
    ("orders", orders_work, "ID", "order_id"),
    ("orders", orders_work, "Customer: ID", "customer_id"),
    ("orders", orders_work, "Line items: ID", "order_line_id"),
    ("orders", orders_work, "Line items / Product: ID", "product_id"),
    ("orders", orders_work, "Line items / Variant: ID", "variant_id"),
    ("orders", orders_work, "Fulfillments: ID", "fulfillment_id"),

    ("customers", customers_work, "ID", "customer_id"),
    ("products", products_work, "ID", "product_id"),

    ("variants", variants_work, "ID", "variant_id"),
    ("variants", variants_work, "Product: ID", "product_id"),
    ("variants", variants_work, "Inventory item: ID", "inventory_item_id"),

    ("inventory", inventory_work, "ID", "inventory_item_id"),
    ("inventory", inventory_work, "Variant: ID", "variant_id"),
    ("inventory", inventory_work, "Inventory levels / Location: ID", "location_id"),
    ("inventory", inventory_work, "Inventory levels: ID", "inventory_level_id")
]

check_results = []

for table_name, df, raw_column, normalized_column in id_checks:
    failed = df[raw_column].notna() & df[normalized_column].isna() # this identifies rows where Shopify provided a GID but normalization lost it
    failed_count = int(failed.sum())

    check_results.append({
        "table": table_name,
        "raw_column": raw_column,
        "normalized_column": normalized_column,
        "rows": len(df),
        "raw_non_null": int(df[raw_column].notna().sum()),
        "normalized_non_null": int(df[normalized_column].notna().sum()),
        "failed": failed_count,
        "status": "PASS" if failed_count == 0 else "FAIL",
    })

# Dictionaries become a DataFrame, producing an auditable QA table
check_results = pd.DataFrame(check_results)
display(check_results)

failed_checks = check_results.loc[check_results["failed"] > 0]

assert failed_checks.empty, (
    "Some non-null Shopify GIDs failed normalization:\n"
    + failed_checks.to_string(index=False)
)

# Validate the composite primary key for the current Inventory Level snapshot.
inventory_level_null_keys = int(
    inventory_work[inventory_level_pk].isna().any(axis=1).sum()
)
inventory_level_duplicate_keys = int(
    inventory_work.duplicated(inventory_level_pk, keep=False).sum()
)

inventory_level_pk_check = pd.DataFrame([{
    "table": "inventory_levels",
    "primary_key": " + ".join(inventory_level_pk),
    "rows": len(inventory_work),
    "null_key_rows": inventory_level_null_keys,
    "duplicate_key_rows": inventory_level_duplicate_keys,
    "status": (
        "PASS"
        if inventory_level_null_keys == 0 and inventory_level_duplicate_keys == 0
        else "FAIL"
    ),
}])

display(inventory_level_pk_check)

assert inventory_level_null_keys == 0, "Inventory Level PK contains nulls"
assert inventory_level_duplicate_keys == 0, "Inventory Level PK is not unique"

In [ ]:
print("Rows:", len(orders_work))
print("Orders:", orders_work["order_id"].nunique())
print("Order lines:", orders_work["order_line_id"].nunique())
print("Null line IDs:", orders_work["order_line_id"].isna().sum())
print("Duplicate line IDs:", orders_work["order_line_id"].duplicated().sum())

In [ ]:
duplicate_mask = orders_work.duplicated(
    "order_line_id",
    keep=False,
)

duplicate_order_lines = (
    orders_work[duplicate_mask]
    .sort_values(["order_line_id", "order_id"])
)

print(
    f"{len(duplicate_order_lines):,} rows across "
    f"{duplicate_order_lines['order_line_id'].nunique():,} "
    "duplicated order-line IDs"
)

display(duplicate_order_lines.head(50))

In [ ]:
variation_by_line = (
    duplicate_order_lines
    .groupby("order_line_id")
    .nunique(dropna=False)
)

varying_columns = (
    variation_by_line.gt(1)
    .sum()
    .sort_values(ascending=False)
)

display(
    varying_columns[
        varying_columns > 0
    ].to_frame("duplicate_groups_with_variation")
)

In [ ]:
order_line_columns = [
    "order_line_id",
    "order_id",
    "product_id",
    "variant_id",
    "Line items: Quantity",
    "Line items: Unfulfilled quantity",
    "Line items: Fulfillable quantity",
    "Line items: Title",
    "Line items: Variant title",
    "Line items: Shop original unit price",
    "Line items: Shop original unit price currency",
    "Line items: Shop original total",
    "Line items: Shop original total currency",
    "Line items: Shop discounted total",
    "Line items: Shop discounted total currency",
    "Line items: Shop total discount",
    "Line items: Shop total discount currency",
    "Line items: Taxable",
    "Line items: Requires shipping",
]

In [ ]:
# Remove identical rows and validate
order_lines_clean = (
    orders_work[order_line_columns]
    .drop_duplicates()
    .copy()
)

assert order_lines_clean["order_line_id"].notna().all(), (
    "Order-line IDs contain nulls"
)

assert order_lines_clean["order_line_id"].is_unique, (
    "Some order lines have conflicting retained attributes"
)

In [ ]:
fulfillments_clean = (
    orders_work[
        ["fulfillment_id", "order_id"]
    ]
    .dropna(subset=["fulfillment_id"])
    .drop_duplicates()
    .copy()
)

In [ ]:
print("Rows:", len(fulfillments_clean))
print("Null fulfillment IDs:", fulfillments_clean["fulfillment_id"].isna().sum())
print("Duplicate fulfillment IDs:", fulfillments_clean["fulfillment_id"].duplicated().sum())
print("Null order IDs:", fulfillments_clean["order_id"].isna().sum())

assert fulfillments_clean["fulfillment_id"].notna().all()
assert fulfillments_clean["fulfillment_id"].is_unique
assert fulfillments_clean["order_id"].notna().all()

In [ ]:
order_columns = [
    "order_id",
    "customer_id",
    "Name",
    "Created at",
    "Updated at",
    "Processed at",
    "Cancelled at",
    "Shipping address: City",
    "Shipping address: Province", 
    "Shipping address: Country",
    "Display financial status",
    "Display fulfillment status",
    "Payment gateways",
    "Currency code",
    "Presentment currency",
    "Shop subtotal price",
    "Shop subtotal price currency",
    "Shop total shipping price",
    "Shop total shipping price currency",
    "Shop total tax",
    "Shop total tax currency",
    "Shop total discounts",
    "Shop total discounts currency",
    "Shop total price",
    "Shop total price currency",
    "Test",
]

In [ ]:
# Create new orders table using order level fields from order_columns
orders_clean = (
    orders_work[order_columns]
    .drop_duplicates()
    .copy()
)

# test row counts
print("Orders:", len(orders_clean))
print("Null order IDs:", orders_clean["order_id"].isna().sum())
print("Duplicate order IDs:", orders_clean["order_id"].duplicated().sum())

# lock in contract
assert orders_clean["order_id"].notna().all()
assert orders_clean["order_id"].is_unique


In [ ]:
print(
    "Order-line orphan rows:",
    (~order_lines_clean["order_id"].isin(orders_clean["order_id"])).sum()
)

print(
    "Fulfillment orphan rows:",
    (~fulfillments_clean["order_id"].isin(orders_clean["order_id"])).sum()
)

assert order_lines_clean["order_id"].isin(
    orders_clean["order_id"]
).all()

assert fulfillments_clean["order_id"].isin(
    orders_clean["order_id"]
).all()

In [ ]:
for column in customers_work.columns:
    print(column)

In [ ]:
customer_columns = [
    "customer_id", 
    "First name",
    "Last name", 
    "Lifetime duration", 
    "Email", 
    "Phone", 
    "Created at",
    "Number of orders",
    "Amount spent", 
    "Updated at",
    "State", 
    "Tags"
]

In [ ]:
# Create customers clean table using customer level columns from customers_work; customer_columns variable
customers_clean = (
    customers_work[customer_columns]
    .drop_duplicates()
    .copy()
)

# Test customer row count; whether one customer_id gives one customer row
print("Customers:", len(customers_clean))
print("Null customer_id count:", customers_clean["customer_id"].isna().sum())
print("Duplicate customer_id count:", customers_clean["customer_id"].duplicated().sum())

assert customers_clean["customer_id"].notna().all()
assert customers_clean["customer_id"].is_unique

In [ ]:
# Validate relationship between orders_clean and customers_clean
order_customer_ids = orders_clean["customer_id"]

print("Orders with null customer_id:", orders_clean["customer_id"].isna().sum())
print("Orders with unmatched customer_id:", (~order_customer_ids.dropna().isin(customers_clean["customer_id"])).sum())

assert order_customer_ids.dropna().isin(customers_clean["customer_id"]).all()

In [ ]:
for column in products_work.columns:
    print(column)

In [ ]:
product_columns = [
    "product_id",
    "Title",
    "Vendor",
    "Product type",
    "Status",
    "Tags",
    "Created at",
    "Updated at",
    "Published at"    
]

In [ ]:
products_clean = (
    products_work[product_columns]
    .drop_duplicates()
    .copy()
)

print("Product row count:", len(products_clean))
print("Products with null product_id:", products_clean["product_id"].isna().sum())
print("Duplicated product_id count:", products_clean["product_id"].duplicated().sum())

assert products_clean["product_id"].notna().all()
assert products_clean["product_id"].is_unique

Count order lines with no product_id.
Then, among order lines that do have product_id, count how many point to products that are missing from products_clean.

In [ ]:
order_line_product_ids = order_lines_clean["product_id"] # Which product, if any, each order line is connected to

# Identify nulls -- then check if non-null product_id in order_lines_clean match products_clean product_id
print("Null product_id in order_lines_clean count:", order_lines_clean["product_id"].isna().sum())
print("Unmatched product_id count:", (~order_line_product_ids.dropna().isin(products_clean["product_id"])).sum()) 

assert order_line_product_ids.dropna().isin(products_clean["product_id"]).all()

In [ ]:
for column in variants_work.columns:
    print(column)

In [ ]:
variant_columns = [
    "variant_id",
    "product_id",
    "inventory_item_id",
    "Title",
    "Created at",
    "Updated at",
    "SKU",
    "Price",
    "Inventory item: Unit cost",
    "Compare-at price",
    "Taxable",
    "Available for sale"
]

In [ ]:
variants_clean = (
    variants_work[variant_columns]
    .drop_duplicates()
    .copy()
)

print("Variants count:", len(variants_clean))
print("Null variant_id count:", variants_clean["variant_id"].isna().sum())
print("Duplicated variant_id count:", variants_clean["variant_id"].duplicated().sum())

assert variants_clean["variant_id"].notna().all()
assert variants_clean["variant_id"].is_unique

Count variant rows with no product_id. Then, of the rows that do have product_id, count how many point to products that are missing from products_clean

In [ ]:
variant_product_ids = variants_clean["product_id"]

print("Null product_id in variants_clean count:", variants_clean["product_id"].isna().sum())
print("Unmatched product_id count:", (~variant_product_ids.dropna().isin(products_clean["product_id"])).sum())

assert variant_product_ids.dropna().isin(products_clean["product_id"]).all()

In [ ]:
order_line_variant_ids = order_lines_clean["variant_id"]

print("Null variant_id in order_line_clean count:", order_lines_clean["variant_id"].isna().sum())
print("Unmatched variant_id count:", (~order_line_variant_ids.dropna().isin(variants_clean["variant_id"])).sum())

assert order_line_variant_ids.dropna().isin(variants_clean["variant_id"]).all()